# Colab T4 실험
런타임을 T4 GPU로 설정하고 저장소와 데이터 경로를 바꾸세요. 숫자는 예비 실험용이며 본 실험 전 공통 확정합니다. 체크포인트는 Drive에 저장합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Drive에 코드 폴더와 LOL 데이터를 준비한 후 경로 수정
from pathlib import Path
import shutil
PROJECT = Path('/content/drive/MyDrive/Image_processing_final_Project')
DATA_SOURCE = Path('/content/drive/MyDrive/data/LOL')
DATA = Path('/content/LOL')
assert PROJECT.is_dir() and DATA_SOURCE.is_dir(), 'Drive 경로를 확인하세요'
shutil.copytree(DATA_SOURCE, DATA, dirs_exist_ok=True)
%cd $PROJECT
%pip install -r requirements.txt


In [ ]:
# TASK-003: Colab T4 GPU 및 환경 버전 확인
import platform, subprocess, sys
import numpy, PIL, torch

assert torch.cuda.is_available(), 'GPU 런타임(T4)을 설정하세요: 런타임 > 런타임 유형 변경 > T4 GPU'
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name)
assert 'T4' in gpu_name.upper(), f'T4 GPU가 아닙니다: {gpu_name}'
print('Python:', sys.version.split()[0], f'({platform.platform()})')
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('numpy:', numpy.__version__, '| Pillow:', PIL.__version__)
print(subprocess.check_output(['nvidia-smi'], text=True))


In [ ]:
import subprocess, sys, torch
assert torch.cuda.is_available(), 'GPU 런타임을 설정하세요'
gpu_name = torch.cuda.get_device_name(0)
assert 'T4' in gpu_name.upper(), f'T4 GPU가 아닙니다: {gpu_name}'
print(gpu_name)
RUNS = '/content/drive/MyDrive/lowlight_runs'
EVAL_OUT = '/content/drive/MyDrive/lowlight_evaluation'
EPOCHS, CROP, BATCH = 10, 128, 8  # 예비 측정 후 공통 확정
SEED, SPLIT_SEED = 0, 0
MODULE_CONFIG = None  # TASK-019에서 별도 JSON 확정 시 경로 지정 (예: 'module_config.json')
common = ['--low-dir',str(DATA/'our485/low'),'--high-dir',str(DATA/'our485/high'),
          '--test-low-dir',str(DATA/'eval15/low'),'--test-high-dir',str(DATA/'eval15/high'),
          '--epochs',str(EPOCHS),'--lr','0.0001','--crop-size',str(CROP),'--batch-size',str(BATCH),
          '--seed',str(SEED),'--split-seed',str(SPLIT_SEED),'--max_hours','3','--output',RUNS]
if MODULE_CONFIG:
    common += ['--module-config', str(MODULE_CONFIG)]


In [ ]:
# TASK-009: 실데이터 한 배치 CUDA forward→loss→backward→step 점검 (본 학습 아님, 체크포인트 저장 없음)
# 위의 Drive·데이터 복사 셀과 공통 설정 셀(DATA, SEED, SPLIT_SEED, CROP, BATCH)을 먼저 실행하세요.
import json, torch
from dataset import create_lol_dataloaders
from model import ZeroDCE
from loss import ZeroDCELoss

assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0).upper(), 'T4 GPU 런타임이 아닙니다'
torch.manual_seed(SEED)
loaders = create_lol_dataloaders(low_dir=DATA/'our485/low', high_dir=DATA/'our485/high',
    test_low_dir=DATA/'eval15/low', test_high_dir=DATA/'eval15/high', val_ratio=0.1,
    seed=SEED, split_seed=SPLIT_SEED, crop_size=CROP, batch_size=BATCH, num_workers=0)
low = next(iter(loaders['train'])).cuda()          # 학습 배치는 low만 (high 미사용)
model = ZeroDCE('none').cuda()
objective = ZeroDCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)  # train.py와 동일
before = [p.detach().clone() for p in model.parameters()]
torch.cuda.reset_peak_memory_stats()

optimizer.zero_grad(set_to_none=True)
enhanced, curves, _ = model(low)
loss, parts = objective(low, enhanced, curves)
loss.backward()
grads_finite = all(bool(torch.isfinite(p.grad).all()) for p in model.parameters())
grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1))  # clip 전 노름
optimizer.step()
torch.cuda.synchronize()
changed = sum(not torch.equal(b, p.detach()) for b, p in zip(before, model.parameters()))

report = {
    'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': torch.version.cuda,
    'split_counts': {k: len(v.dataset) for k, v in loaders.items()},
    'model_device': str(next(model.parameters()).device), 'input_device': str(low.device),
    'input_shape': list(low.shape), 'input_dtype': str(low.dtype),
    'input_range': [float(low.min()), float(low.max())],
    'output_shape': list(enhanced.shape), 'curves_shape': list(curves.shape),
    'loss': float(loss), 'loss_parts': {k: float(v) for k, v in parts.items()},
    'loss_finite': bool(torch.isfinite(loss)), 'grads_finite': grads_finite,
    'grad_norm_before_clip': grad_norm,
    'parameters_changed': f'{changed}/{len(before)}',
    'peak_memory_mb': round(torch.cuda.max_memory_allocated() / 2**20, 1),
}
print(json.dumps(report, indent=2, ensure_ascii=False))
assert report['model_device'].startswith('cuda') and report['input_device'].startswith('cuda')
assert report['loss_finite'] and grads_finite and changed == len(before), 'TASK-009 실패: 위 출력을 확인하세요'
print('TASK-009 통과: 출력 전체를 results/TASK-009-gpu-smoke.txt에 기록하세요.')


In [ ]:
# 우선 baseline을 실행하여 시간을 확인하세요. 재개 시 같은 명령에 --resume 추가
subprocess.run([sys.executable,'train.py',*common,'--mode','none'],check=True)


In [ ]:
# 공통 설정 확정 후 실행. 설정을 바꾼 예비 실험과 본 실험은 RUNS 경로를 분리하세요.
for mode in ['bright','noise','both']:
    subprocess.run([sys.executable,'train.py',*common,'--mode',mode],check=True)


In [ ]:
# 튜닝에는 val만 사용
subprocess.run([sys.executable,'eval.py','--runs',RUNS,'--split','val',
                '--output',EVAL_OUT],check=True)


In [ ]:
# 설정 확정 후 최종 test. 기본 모델도 같은 조건으로 직접 학습되어 있어야 합니다.
subprocess.run([sys.executable,'eval.py','--runs',RUNS,'--split','test',
                '--output',EVAL_OUT],check=True)
